# Aggregations

Aggregating means collecting many rows into a summary: a total, an average, a count. This notebook goes through the ways Spark lets you do that, from a single number for the whole DataFrame to group-level summaries, window functions, rollups, cubes and pivots. Most examples are followed by the same query in Spark SQL.

The next cell finds your cluster's bucket for you, so there is nothing to edit.

In [1]:
# the following line gets the bucket name attached to our cluster
bucket = spark._jsc.hadoopConfiguration().get("fs.gs.system.bucket")

# specifying the path to our bucket where the data is located (no need to edit this path anymore)
data = "gs://" + bucket + "/notebooks/jupyter/Big-Data-Analytics-for-Business/data/"
print(data)

gs://qst843/notebooks/jupyter/Big-Data-Analytics-for-Business/data/


## Data import

We will use the online retail data from a UK gift shop. There are a bit over 300 CSV files in `retail-data/by-day/`, one per day of transactions between December 2010 and December 2011. Spark reads all of them into one DataFrame when we pass a wildcard:

In [2]:
df = spark.read.format("csv")\
  .option("header", "true")\
  .option("inferSchema", "true")\
  .load(data + "retail-data/by-day/*.csv")

Here's a sample of the data:

In [3]:
df.show(5, False)

df.printSchema()

+---------+---------+-------------------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|Description                    |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+---------+---------+-------------------------------+--------+-------------------+---------+----------+--------------+
|580538   |23084    |RABBIT NIGHT LIGHT             |48      |2011-12-05 08:38:00|1.79     |14075.0   |United Kingdom|
|580538   |23077    |DOUGHNUT LIP GLOSS             |20      |2011-12-05 08:38:00|1.25     |14075.0   |United Kingdom|
|580538   |22906    |12 MESSAGE CARDS WITH ENVELOPES|24      |2011-12-05 08:38:00|1.65     |14075.0   |United Kingdom|
|580538   |21914    |BLUE HARMONICA IN BOX          |24      |2011-12-05 08:38:00|1.25     |14075.0   |United Kingdom|
|580538   |22467    |GUMBALL COAT RACK              |6       |2011-12-05 08:38:00|2.55     |14075.0   |United Kingdom|
+---------+---------+---------------------------

A few columns need fixing before we start:

* `InvoiceDate` comes in as a timestamp. We only care about the day, so we turn it into a date.
* `CustomerID` was inferred as a double, which is why it prints as `14075.0`. An ID is a label, not a number, so we cast it to a whole number first and then to a string.
* We also add a `Revenue` column (quantity times unit price), since money is what most business questions are about.

In [4]:
from pyspark.sql.functions import col, to_date, round

df = df.withColumn("InvoiceDate", to_date(col("InvoiceDate")))\
  .withColumn("CustomerID", col("CustomerID").cast("long").cast("string"))\
  .withColumn("Quantity", col("Quantity").cast("long"))\
  .withColumn("Revenue", round(col("Quantity") * col("UnitPrice"), 2))

df.createOrReplaceTempView("dfTable")

df.show(5)
df.printSchema()

+---------+---------+--------------------+--------+-----------+---------+----------+--------------+-------+
|InvoiceNo|StockCode|         Description|Quantity|InvoiceDate|UnitPrice|CustomerID|       Country|Revenue|
+---------+---------+--------------------+--------+-----------+---------+----------+--------------+-------+
|   580538|    23084|  RABBIT NIGHT LIGHT|      48| 2011-12-05|     1.79|     14075|United Kingdom|  85.92|
|   580538|    23077| DOUGHNUT LIP GLOSS |      20| 2011-12-05|     1.25|     14075|United Kingdom|   25.0|
|   580538|    22906|12 MESSAGE CARDS ...|      24| 2011-12-05|     1.65|     14075|United Kingdom|   39.6|
|   580538|    21914|BLUE HARMONICA IN...|      24| 2011-12-05|     1.25|     14075|United Kingdom|   30.0|
|   580538|    22467|   GUMBALL COAT RACK|       6| 2011-12-05|     2.55|     14075|United Kingdom|   15.3|
+---------+---------+--------------------+--------+-----------+---------+----------+--------------+-------+
only showing top 5 rows

roo

## Caching

Every action re-reads the 300 files from Cloud Storage unless we tell Spark to keep the data in memory. `cache()` does exactly that.

**Note:** `cache()` is lazy. Nothing is stored until the first action runs on the DataFrame.

In [5]:
df.cache()

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: bigint, InvoiceDate: date, UnitPrice: double, CustomerID: string, Country: string, Revenue: double]

The simplest aggregation is the `count` method. Used this way it is an action, so it runs right away and returns a number. Because it is the first action after `cache()`, it also fills the cache. `%time` shows how long the cell took:

In [6]:
%time df.count()

CPU times: user 81.4 ms, sys: 20.9 ms, total: 102 ms
Wall time: 17.4 s


541909

Run the same count again. This time the data comes from memory, so it should be much faster:

In [7]:
%time df.count()

CPU times: user 32.1 ms, sys: 4.21 ms, total: 36.3 ms
Wall time: 484 ms


541909

When you are done with a cached DataFrame, you can free the memory with `df.unpersist()`.

## Aggregation functions

Most aggregation functions live in `pyspark.sql.functions`. You use them inside `select()` (or `agg()`, which we will see shortly), and they return a DataFrame, so they are lazy like any other transformation.

**Note:** importing `sum`, `min`, `max` or `round` from `pyspark.sql.functions` replaces Python's built-in functions with the same names for the rest of the notebook. That is fine here, but in your own code you may prefer `import pyspark.sql.functions as F` and then write `F.sum(...)`.

### `count`

`count` as a function counts the non-null values in a column. To count every row, use `count("*")` (or `count(1)`):

In [8]:
from pyspark.sql.functions import count

df.select(count("*"), count("StockCode"), count("CustomerID")).show()

+--------+----------------+-----------------+
|count(1)|count(StockCode)|count(CustomerID)|
+--------+----------------+-----------------+
|  541909|          541909|           406829|
+--------+----------------+-----------------+



`count("*")` counts all rows, including rows with nulls. Counting a single column skips nulls, which is why `CustomerID` comes out about 135,000 lower: those transactions have no customer attached.

In SQL
```sql
SELECT COUNT(*), COUNT(StockCode), COUNT(CustomerID) FROM dfTable
```

### `countDistinct` and `approx_count_distinct`

Often you want the number of unique values rather than the number of rows:

In [9]:
from pyspark.sql.functions import countDistinct

df.select(countDistinct("StockCode")).show()

+-------------------------+
|count(DISTINCT StockCode)|
+-------------------------+
|                     4070|
+-------------------------+



In SQL
```sql
SELECT COUNT(DISTINCT StockCode) FROM dfTable
```

An exact distinct count has to keep track of every value it has seen, which gets expensive on very large data. `approx_count_distinct` gives an estimate that is much cheaper to compute. The second argument is the relative standard deviation you will accept (here 5%):

In [10]:
from pyspark.sql.functions import approx_count_distinct

df.select(approx_count_distinct("StockCode", 0.05)).show()

+--------------------------------+
|approx_count_distinct(StockCode)|
+--------------------------------+
|                            3804|
+--------------------------------+



The estimate is 3,804 against the true 4,070, about 6.5% off. The 5% is a standard deviation, not a hard limit, so an estimate can miss by a bit more. On billions of rows, that trade is often worth it.

### `min`, `max`, and `sum`

In [11]:
from pyspark.sql.functions import min, max, sum

df.select(min("Quantity"), max("Quantity"), sum("Quantity"), sum("Revenue")).show()

+-------------+-------------+-------------+-----------------+
|min(Quantity)|max(Quantity)|sum(Quantity)|     sum(Revenue)|
+-------------+-------------+-------------+-----------------+
|       -80995|        80995|      5176450|9747747.929999461|
+-------------+-------------+-------------+-----------------+



The minimum quantity is negative. Invoices that start with "C" are cancellations, and they show up as negative quantities. Keep that in mind when you read totals.

### `avg`

`avg` and `mean` are the same function. Below we compute the average three ways and check they agree:

In [12]:
from pyspark.sql.functions import avg, expr

df.select(
    count("Quantity").alias("total_transactions"),
    sum("Quantity").alias("total_purchases"),
    avg("Quantity").alias("avg_purchases"),
    expr("mean(Quantity)").alias("mean_purchases"))\
  .selectExpr(
    "total_purchases/total_transactions",
    "avg_purchases",
    "mean_purchases").show()

+--------------------------------------+----------------+----------------+
|(total_purchases / total_transactions)|   avg_purchases|  mean_purchases|
+--------------------------------------+----------------+----------------+
|                      9.55224954743324|9.55224954743324|9.55224954743324|
+--------------------------------------+----------------+----------------+



### Variance and standard deviation

Both measure how spread out the values are around the mean. `variance` and `stddev` use the *sample* formulas by default:

In [13]:
from pyspark.sql.functions import variance, stddev

df.select(variance("Quantity"), stddev("Quantity")).show()

+------------------+-----------------+
|var_samp(Quantity)| stddev(Quantity)|
+------------------+-----------------+
|47559.391409298456|218.0811578502335|
+------------------+-----------------+



You can ask for the sample or population versions explicitly:

In [14]:
from pyspark.sql.functions import var_pop, stddev_pop
from pyspark.sql.functions import var_samp, stddev_samp

df.select(var_pop("Quantity"), var_samp("Quantity"),
  stddev_pop("Quantity"), stddev_samp("Quantity")).show()

+------------------+------------------+--------------------+---------------------+
| var_pop(Quantity)|var_samp(Quantity)|stddev_pop(Quantity)|stddev_samp(Quantity)|
+------------------+------------------+--------------------+---------------------+
|47559.303646608765|47559.391409298456|   218.0809566344773|    218.0811578502335|
+------------------+------------------+--------------------+---------------------+



In SQL
```sql
SELECT var_pop(Quantity), var_samp(Quantity),
  stddev_pop(Quantity), stddev_samp(Quantity)
FROM dfTable
```

### Medians and percentiles

The average is easily pulled around by a few huge orders (remember that 80,995 quantity). The median is not. `percentile_approx` returns approximate percentiles, and since Spark 3.4 there is also a `median` function:

In [15]:
from pyspark.sql.functions import median, percentile_approx

df.select(
    median("UnitPrice").alias("median_price"),
    percentile_approx("UnitPrice", [0.25, 0.5, 0.75]).alias("price_quartiles"),
    avg("UnitPrice").alias("avg_price")).show(truncate=False)

+------------+------------------+-----------------+
|median_price|price_quartiles   |avg_price        |
+------------+------------------+-----------------+
|2.08        |[1.25, 2.08, 4.13]|4.611113626089641|
+------------+------------------+-----------------+



### Covariance and correlation

These compare two columns. `corr` is the Pearson correlation, which is always between -1 and +1. Covariance depends on the scale of the inputs. There are `covar_samp` and `covar_pop` versions:

In [16]:
from pyspark.sql.functions import corr, covar_samp

df.select(corr("UnitPrice", "Quantity"), covar_samp("UnitPrice", "Quantity")).show()

+-------------------------+-------------------------------+
|corr(UnitPrice, Quantity)|covar_samp(UnitPrice, Quantity)|
+-------------------------+-------------------------------+
|     -0.00123492454487...|            -26.058761257936645|
+-------------------------+-------------------------------+



A correlation this close to zero means there is no linear relationship between price and quantity in this data.

### Aggregating to complex types

You can also collect values into a list (`collect_list`) or a set of unique values (`collect_set`). This is handy later, for example to pass all the countries a product sells in to another function:

In [17]:
from pyspark.sql.functions import collect_set, size

df.select(size(collect_set("Country")).alias("n_countries"),
          collect_set("Country").alias("countries")).show(truncate=80)

+-----------+--------------------------------------------------------------------------------+
|n_countries|                                                                       countries|
+-----------+--------------------------------------------------------------------------------+
|         38|[Portugal, Italy, Finland, Germany, RSA, Canada, Switzerland, Greece, Spain, ...|
+-----------+--------------------------------------------------------------------------------+



## Grouping

So far each aggregation produced one row for the whole DataFrame. Usually we want one row per group: per invoice, per customer, per country.

Grouping happens in two steps. First `groupBy()` names the column(s) to group on and returns a `GroupedData` object. Then we say which aggregation(s) to run, and that returns a DataFrame:

In [18]:
df.groupBy("InvoiceNo", "CustomerID")

GroupedData[grouping expressions: [InvoiceNo, CustomerID], value: [InvoiceNo: string, StockCode: string ... 7 more fields], type: GroupBy]

In [19]:
df.groupBy("InvoiceNo", "CustomerID").count().show(5)

+---------+----------+-----+
|InvoiceNo|CustomerID|count|
+---------+----------+-----+
|  C581460|     13078|    2|
|   579273|     15197|   17|
|   570353|     17706|    4|
|   580739|      NULL|    2|
|   567695|      NULL|    1|
+---------+----------+-----+
only showing top 5 rows



In SQL
```sql
SELECT InvoiceNo, CustomerID, count(*) FROM dfTable GROUP BY InvoiceNo, CustomerID
```

### Grouping with expressions

To run more than one aggregation, or to name the results, pass the functions to `agg()`. You can mix functions and SQL strings in `expr()`:

In [20]:
from pyspark.sql.functions import count, expr

df.groupBy("InvoiceNo")\
  .agg(count("Quantity").alias("quan"),
       expr("count(Quantity) as quan2")
      ).show(5)

+---------+----+-----+
|InvoiceNo|quan|quan2|
+---------+----+-----+
|   578057|  28|   28|
|  C578132|   1|    1|
|   576112|  20|   20|
|   580739|   2|    2|
|   567695|   1|    1|
+---------+----+-----+
only showing top 5 rows



A dictionary also works when you only need one function per column:

In [21]:
df.groupBy("InvoiceNo").agg({"Quantity": "avg", "UnitPrice": "max"}).show(5)

+---------+--------------+-----------------+
|InvoiceNo|max(UnitPrice)|    avg(Quantity)|
+---------+--------------+-----------------+
|   578057|         16.95|4.607142857142857|
|  C578132|          0.85|             -1.0|
|   576112|          8.25|             10.9|
|   580739|           2.1|              2.5|
|   567695|           0.0|            -42.0|
+---------+--------------+-----------------+
only showing top 5 rows



Here is a more realistic question: which countries bring in the most revenue, from how many customers? After the `agg()` we can filter and sort the result like any DataFrame. A filter after a group-by is what SQL calls `HAVING`.

In [22]:
from pyspark.sql.functions import desc

df.groupBy("Country")\
  .agg(round(sum("Revenue"), 0).alias("revenue"),
       countDistinct("CustomerID").alias("customers"),
       countDistinct("InvoiceNo").alias("invoices"))\
  .where("revenue > 50000")\
  .orderBy(desc("revenue"))\
  .show()

+--------------+---------+---------+--------+
|       Country|  revenue|customers|invoices|
+--------------+---------+---------+--------+
|United Kingdom|8187806.0|     3950|   23494|
|   Netherlands| 284662.0|        9|     101|
|          EIRE| 263277.0|        3|     360|
|       Germany| 221698.0|       95|     603|
|        France| 197404.0|       87|     461|
|     Australia| 137077.0|        9|      69|
|   Switzerland|  56385.0|       21|      74|
|         Spain|  54775.0|       31|     105|
+--------------+---------+---------+--------+



In SQL
```sql
SELECT Country,
       round(sum(Revenue), 0) AS revenue,
       count(DISTINCT CustomerID) AS customers,
       count(DISTINCT InvoiceNo) AS invoices
FROM dfTable
GROUP BY Country
HAVING revenue > 50000
ORDER BY revenue DESC
```

Spark 3.4 and 3.5 added a few handy aggregate functions. `count_if` counts the rows where a condition is true, and `max_by` returns the value of one column at the row where another column is largest. Below, for each country, we count cancelled lines and find the product with the single biggest sale:

In [23]:
from pyspark.sql.functions import count_if, max_by

df.groupBy("Country")\
  .agg(count_if(col("InvoiceNo").startswith("C")).alias("cancelled_lines"),
       max_by("Description", "Revenue").alias("biggest_sale_item"))\
  .orderBy(desc("cancelled_lines"))\
  .show(5, False)

+--------------+---------------+----------------------------------+
|Country       |cancelled_lines|biggest_sale_item                 |
+--------------+---------------+----------------------------------+
|United Kingdom|7856           |PAPER CRAFT , LITTLE BIRDIE       |
|Germany       |453            |REGENCY CAKESTAND 3 TIER          |
|EIRE          |302            |RED RETROSPOT CAKE STAND          |
|France        |149            |Manual                            |
|USA           |112            |SET 6 SCHOOL MILK BOTTLES IN CRATE|
+--------------+---------------+----------------------------------+
only showing top 5 rows



The UK's biggest sale, `PAPER CRAFT , LITTLE BIRDIE`, is the 80,995-unit order we saw earlier. It was cancelled the same day.

## Window functions

A group-by squeezes each group into one row. A window function also looks at a group of rows (the *window*), but it returns a value for **every** row. That lets you put a per-customer total next to each transaction, rank rows inside a group, or compute a running total.

The SQL query below adds `overall_dataset_Q`, the total quantity of the whole dataset, to every row:

In [24]:
spark.sql("""
SELECT CustomerID, InvoiceDate, Quantity,

  sum(Quantity) OVER () as overall_dataset_Q

FROM dfTable WHERE CustomerID IS NOT NULL
ORDER BY CustomerID, InvoiceDate DESC
""").show(5)

26/10/17 18:17:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+--------+-----------------+
|CustomerID|InvoiceDate|Quantity|overall_dataset_Q|
+----------+-----------+--------+-----------------+
|     12346| 2011-01-18|   74215|          4906888|
|     12346| 2011-01-18|  -74215|          4906888|
|     12347| 2011-12-07|      12|          4906888|
|     12347| 2011-12-07|      24|          4906888|
|     12347| 2011-12-07|       6|          4906888|
+----------+-----------+--------+-----------------+
only showing top 5 rows



The `OVER()` clause turns `sum()` into a window function. Inside it you can:

* split the rows into groups with `PARTITION BY`
* order the rows inside each group with `ORDER BY`

We left `OVER()` empty, so the window is the whole dataset. That is why Spark printed a `WARN` about moving all data to a single partition: with no `PARTITION BY`, one task has to see every row. On big data, avoid empty windows.

Adding `PARTITION BY` gives us a total per customer and a total per customer per day:

In [25]:
spark.sql("""
SELECT CustomerID, InvoiceDate, Quantity,

  sum(Quantity) OVER () as overall_dataset_Q,

  sum(Quantity) OVER (PARTITION BY CustomerID) as total_Q,

  sum(Quantity) OVER (PARTITION BY CustomerID, InvoiceDate) as total_Q_by_date

FROM dfTable WHERE CustomerID IS NOT NULL
ORDER BY CustomerID, InvoiceDate DESC
""").show()

26/10/17 18:17:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+--------+-----------------+-------+---------------+
|CustomerID|InvoiceDate|Quantity|overall_dataset_Q|total_Q|total_Q_by_date|
+----------+-----------+--------+-----------------+-------+---------------+
|     12346| 2011-01-18|   74215|          4906888|      0|              0|
|     12346| 2011-01-18|  -74215|          4906888|      0|              0|
|     12347| 2011-12-07|      12|          4906888|   2458|            192|
|     12347| 2011-12-07|       6|          4906888|   2458|            192|
|     12347| 2011-12-07|      24|          4906888|   2458|            192|
|     12347| 2011-12-07|      24|          4906888|   2458|            192|
|     12347| 2011-12-07|      24|          4906888|   2458|            192|
|     12347| 2011-12-07|      24|          4906888|   2458|            192|
|     12347| 2011-12-07|      10|          4906888|   2458|            192|
|     12347| 2011-12-07|      12|          4906888|   2458|            192|
|     12347|

Customer 12346 bought 74,215 units and then cancelled them, so their total is 0.

With `ORDER BY` inside the window we can rank the rows in each partition. `RANK()` gives the largest purchase of each customer's day rank 1:

In [26]:
spark.sql("""
SELECT CustomerID, InvoiceDate, Quantity,

  sum(Quantity) OVER (PARTITION BY CustomerID, InvoiceDate) as total_Q_by_date,

  RANK() OVER (PARTITION BY CustomerID, InvoiceDate
               ORDER BY Quantity DESC) as rank

FROM dfTable WHERE CustomerID IS NOT NULL
ORDER BY CustomerID, InvoiceDate DESC, rank
""").show()

+----------+-----------+--------+---------------+----+
|CustomerID|InvoiceDate|Quantity|total_Q_by_date|rank|
+----------+-----------+--------+---------------+----+
|     12346| 2011-01-18|   74215|              0|   1|
|     12346| 2011-01-18|  -74215|              0|   2|
|     12347| 2011-12-07|      24|            192|   1|
|     12347| 2011-12-07|      24|            192|   1|
|     12347| 2011-12-07|      24|            192|   1|
|     12347| 2011-12-07|      24|            192|   1|
|     12347| 2011-12-07|      20|            192|   5|
|     12347| 2011-12-07|      20|            192|   5|
|     12347| 2011-12-07|      16|            192|   7|
|     12347| 2011-12-07|      12|            192|   8|
|     12347| 2011-12-07|      12|            192|   8|
|     12347| 2011-12-07|      10|            192|  10|
|     12347| 2011-12-07|       6|            192|  11|
|     12347| 2011-10-31|      48|            676|   1|
|     12347| 2011-10-31|      36|            676|   2|
|     1234

Look at what `RANK()` does with tied quantities: tied rows share a rank, and the next rank is skipped (1, 2, 2, 4). Two related functions handle ties differently:

* `DENSE_RANK()` also gives ties the same rank, but leaves no gaps (1, 2, 2, 3).
* `ROW_NUMBER()` numbers the rows 1, 2, 3, ... with no ties at all. Tied rows are numbered in an arbitrary order, so the result can change from run to run. If that matters, add a second column to the `ORDER BY` to break ties.

In [27]:
spark.sql("""
SELECT CustomerID, InvoiceDate, Quantity,

  RANK() OVER (PARTITION BY CustomerID, InvoiceDate
               ORDER BY Quantity DESC) as rank,

  DENSE_RANK() OVER (PARTITION BY CustomerID, InvoiceDate
                     ORDER BY Quantity DESC) as d_rank,

  ROW_NUMBER() OVER (PARTITION BY CustomerID, InvoiceDate
                     ORDER BY Quantity DESC) as row_number

FROM dfTable WHERE CustomerID IS NOT NULL
ORDER BY CustomerID, InvoiceDate DESC, row_number
""").show()

+----------+-----------+--------+----+------+----------+
|CustomerID|InvoiceDate|Quantity|rank|d_rank|row_number|
+----------+-----------+--------+----+------+----------+
|     12346| 2011-01-18|   74215|   1|     1|         1|
|     12346| 2011-01-18|  -74215|   2|     2|         2|
|     12347| 2011-12-07|      24|   1|     1|         1|
|     12347| 2011-12-07|      24|   1|     1|         2|
|     12347| 2011-12-07|      24|   1|     1|         3|
|     12347| 2011-12-07|      24|   1|     1|         4|
|     12347| 2011-12-07|      20|   5|     2|         5|
|     12347| 2011-12-07|      20|   5|     2|         6|
|     12347| 2011-12-07|      16|   7|     3|         7|
|     12347| 2011-12-07|      12|   8|     4|         8|
|     12347| 2011-12-07|      12|   8|     4|         9|
|     12347| 2011-12-07|      10|  10|     5|        10|
|     12347| 2011-12-07|       6|  11|     6|        11|
|     12347| 2011-10-31|      48|   1|     1|         1|
|     12347| 2011-10-31|      3

A few points about ranking functions:

* `ROW_NUMBER()`, `RANK()` and `DENSE_RANK()` only work as window functions, so they always need an `OVER` clause.
* They also need an `ORDER BY` inside the window, because the order decides the ranking.

### Running totals and previous rows

When the window has an `ORDER BY`, aggregate functions like `sum()` become cumulative: each row sees the rows up to itself. The frame clause `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` makes this explicit. `LAG()` reaches back to the previous row, which is how you compare a value with the one before it.

Here is each customer's spending by day, with a running total and the number of days since their previous purchase:

In [28]:
spark.sql("""
WITH daily AS (
  SELECT CustomerID, InvoiceDate, round(sum(Revenue), 2) AS revenue
  FROM dfTable
  WHERE CustomerID IS NOT NULL
  GROUP BY CustomerID, InvoiceDate
)
SELECT CustomerID, InvoiceDate, revenue,

  round(sum(revenue) OVER (PARTITION BY CustomerID ORDER BY InvoiceDate
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_revenue,

  datediff(InvoiceDate,
           LAG(InvoiceDate) OVER (PARTITION BY CustomerID ORDER BY InvoiceDate)) AS days_since_last

FROM daily
ORDER BY CustomerID, InvoiceDate
""").show(10)

+----------+-----------+-------+---------------+---------------+
|CustomerID|InvoiceDate|revenue|running_revenue|days_since_last|
+----------+-----------+-------+---------------+---------------+
|     12346| 2011-01-18|    0.0|            0.0|           NULL|
|     12347| 2010-12-07| 711.79|         711.79|           NULL|
|     12347| 2011-01-26| 475.39|        1187.18|             50|
|     12347| 2011-04-07| 636.25|        1823.43|             71|
|     12347| 2011-06-09| 382.52|        2205.95|             63|
|     12347| 2011-08-02| 584.91|        2790.86|             54|
|     12347| 2011-10-31|1294.32|        4085.18|             90|
|     12347| 2011-12-07| 224.82|         4310.0|             37|
|     12348| 2010-12-16|  892.8|          892.8|           NULL|
|     12348| 2011-01-25| 227.44|        1120.24|             40|
+----------+-----------+-------+---------------+---------------+
only showing top 10 rows



The first purchase of each customer has no previous row, so `days_since_last` is `NULL`. `LEAD()` does the same thing looking forward.

### PySpark syntax

Now the same ideas with the DataFrame API. First we build window specifications with `Window`. The `partitionBy` here has nothing to do with how the data is partitioned across the cluster. It only describes how the rows are grouped for the window.

In [29]:
from pyspark.sql.window import Window

windowSpec0 = Window.partitionBy()
windowSpec1 = Window.partitionBy("CustomerID")
windowSpec2 = Window.partitionBy("CustomerID", "InvoiceDate")
windowSpec3 = Window.partitionBy("CustomerID", "InvoiceDate").orderBy(desc("Quantity"))

Then we apply a function `.over()` a window. The result is a column expression we can use in `select()`. We give the variables a `_col` suffix so they do not overwrite the `rank`, `dense_rank` and `row_number` functions themselves (if they did, running the cell a second time would fail).

In [30]:
from pyspark.sql.functions import sum, max, rank, dense_rank, row_number

overall_dataset_Q = sum(col("Quantity")).over(windowSpec0)
total_Q = sum(col("Quantity")).over(windowSpec1)
max_Q = max(col("Quantity")).over(windowSpec1)
total_Q_by_date = sum(col("Quantity")).over(windowSpec2)
rank_col = rank().over(windowSpec3)
d_rank_col = dense_rank().over(windowSpec3)
row_number_col = row_number().over(windowSpec3)

Now we select them next to the original columns:

In [31]:
df.where("CustomerID IS NOT NULL")\
  .select(
    col("CustomerID"),
    col("InvoiceDate"),
    col("Quantity"),
    overall_dataset_Q.alias("overall_dataset_Q"),
    total_Q.alias("total_Q"),
    max_Q.alias("max_Q"),
    total_Q_by_date.alias("total_Q_by_date"),
    rank_col.alias("rank"),
    d_rank_col.alias("d_rank"),
    row_number_col.alias("row_number"))\
  .orderBy("CustomerID", desc("InvoiceDate"), "row_number").show()

26/10/17 18:17:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+--------+-----------------+-------+-----+---------------+----+------+----------+
|CustomerID|InvoiceDate|Quantity|overall_dataset_Q|total_Q|max_Q|total_Q_by_date|rank|d_rank|row_number|
+----------+-----------+--------+-----------------+-------+-----+---------------+----+------+----------+
|     12346| 2011-01-18|   74215|          4906888|      0|74215|              0|   1|     1|         1|
|     12346| 2011-01-18|  -74215|          4906888|      0|74215|              0|   2|     2|         2|
|     12347| 2011-12-07|      24|          4906888|   2458|  240|            192|   1|     1|         1|
|     12347| 2011-12-07|      24|          4906888|   2458|  240|            192|   1|     1|         2|
|     12347| 2011-12-07|      24|          4906888|   2458|  240|            192|   1|     1|         3|
|     12347| 2011-12-07|      24|          4906888|   2458|  240|            192|   1|     1|         4|
|     12347| 2011-12-07|      20|          4906888|   2

### Top N per group

One of the most common uses of `row_number()` is picking the top N rows in each group. Which three products earn the most in each country? First aggregate, then number the products inside each country, then keep numbers 1 to 3:

In [32]:
product_revenue = df.groupBy("Country", "Description")\
  .agg(round(sum("Revenue"), 0).alias("revenue"))

byCountry = Window.partitionBy("Country").orderBy(desc("revenue"))

product_revenue\
  .withColumn("rn", row_number().over(byCountry))\
  .where("rn <= 3")\
  .where(col("Country").isin("France", "Germany", "Japan"))\
  .orderBy("Country", "rn")\
  .show(truncate=False)

+-------+-----------------------------------+-------+---+
|Country|Description                        |revenue|rn |
+-------+-----------------------------------+-------+---+
|France |POSTAGE                            |15065.0|1  |
|France |RABBIT NIGHT LIGHT                 |7275.0 |2  |
|France |REGENCY CAKESTAND 3 TIER           |2582.0 |3  |
|Germany|POSTAGE                            |20821.0|1  |
|Germany|REGENCY CAKESTAND 3 TIER           |8257.0 |2  |
|Germany|ROUND SNACK BOXES SET OF4 WOODLAND |3555.0 |3  |
|Japan  |RABBIT NIGHT LIGHT                 |6088.0 |1  |
|Japan  |ROUND SNACK BOXES SET OF 4 FRUITS  |2221.0 |2  |
|Japan  |RED SPOTTY BISCUIT TIN             |816.0  |3  |
+-------+-----------------------------------+-------+---+



The running total from the SQL section looks like this in PySpark. `rowsBetween` sets the frame, and `lag` gets the previous row:

In [33]:
from pyspark.sql.functions import lag, datediff

daily = df.where("CustomerID IS NOT NULL")\
  .groupBy("CustomerID", "InvoiceDate")\
  .agg(round(sum("Revenue"), 2).alias("revenue"))

byCustomer = Window.partitionBy("CustomerID").orderBy("InvoiceDate")
runningFrame = byCustomer.rowsBetween(Window.unboundedPreceding, Window.currentRow)

daily\
  .withColumn("running_revenue", round(sum("revenue").over(runningFrame), 2))\
  .withColumn("days_since_last", datediff("InvoiceDate", lag("InvoiceDate").over(byCustomer)))\
  .orderBy("CustomerID", "InvoiceDate")\
  .show(10)

+----------+-----------+-------+---------------+---------------+
|CustomerID|InvoiceDate|revenue|running_revenue|days_since_last|
+----------+-----------+-------+---------------+---------------+
|     12346| 2011-01-18|    0.0|            0.0|           NULL|
|     12347| 2010-12-07| 711.79|         711.79|           NULL|
|     12347| 2011-01-26| 475.39|        1187.18|             50|
|     12347| 2011-04-07| 636.25|        1823.43|             71|
|     12347| 2011-06-09| 382.52|        2205.95|             63|
|     12347| 2011-08-02| 584.91|        2790.86|             54|
|     12347| 2011-10-31|1294.32|        4085.18|             90|
|     12347| 2011-12-07| 224.82|         4310.0|             37|
|     12348| 2010-12-16|  892.8|          892.8|           NULL|
|     12348| 2011-01-25| 227.44|        1120.24|             40|
+----------+-----------+-------+---------------+---------------+
only showing top 10 rows



## Rollups

A rollup groups by several columns *and* adds subtotals going up the hierarchy. Rolling up by `InvoiceDate` and then `Country` gives us:

- the total for each country on each date
- the total for each date (all countries)
- the grand total over everything

Rollups mark subtotal rows with `NULL` in the rolled-up columns. To avoid confusing those with real missing values, we first drop the rows that have nulls:

In [34]:
dfNoNull = df.na.drop()
dfNoNull.createOrReplaceTempView("dfNoNull")

In [35]:
rolledUpDF = dfNoNull.rollup("InvoiceDate", "Country")\
  .agg(sum("Quantity"))\
  .orderBy("InvoiceDate")

rolledUpDF.show(10)

+-----------+--------------+-------------+
|InvoiceDate|       Country|sum(Quantity)|
+-----------+--------------+-------------+
|       NULL|          NULL|      4906888|
| 2010-12-01|     Australia|          107|
| 2010-12-01|        Norway|         1852|
| 2010-12-01|          NULL|        24032|
| 2010-12-01|       Germany|          117|
| 2010-12-01|          EIRE|          243|
| 2010-12-01|   Netherlands|           97|
| 2010-12-01|United Kingdom|        21167|
| 2010-12-01|        France|          449|
| 2010-12-02|          EIRE|            4|
+-----------+--------------+-------------+
only showing top 10 rows



The rows with `NULL` in `Country` are the totals for each date across all countries:

In [36]:
rolledUpDF.where("Country IS NULL").where("InvoiceDate IS NOT NULL").show(5)

+-----------+-------+-------------+
|InvoiceDate|Country|sum(Quantity)|
+-----------+-------+-------------+
| 2010-12-01|   NULL|        24032|
| 2010-12-02|   NULL|        20855|
| 2010-12-03|   NULL|        11548|
| 2010-12-05|   NULL|        16394|
| 2010-12-06|   NULL|        16095|
+-----------+-------+-------------+
only showing top 5 rows



And a `NULL` in both columns is the grand total:

In [37]:
rolledUpDF.where("InvoiceDate IS NULL").show()

+-----------+-------+-------------+
|InvoiceDate|Country|sum(Quantity)|
+-----------+-------+-------------+
|       NULL|   NULL|      4906888|
+-----------+-------+-------------+



In SQL
```sql
SELECT InvoiceDate, Country, sum(Quantity)
FROM dfNoNull
GROUP BY ROLLUP(InvoiceDate, Country)
ORDER BY InvoiceDate
```

## Cube

A rollup follows the order of the columns: date first, then country within date. A cube makes subtotals for **every** combination of the columns. With date and country that means:

* the total for each country on each date
* the total for each date across all countries
* the total for each country across all dates
* the grand total

In [38]:
cubedDF = dfNoNull.cube("InvoiceDate", "Country")\
  .agg(sum("Quantity"))\
  .orderBy("InvoiceDate")

cubedDF.show(5)

+-----------+--------------+-------------+
|InvoiceDate|       Country|sum(Quantity)|
+-----------+--------------+-------------+
|       NULL|United Kingdom|      4008533|
|       NULL|        Poland|         3653|
|       NULL|         Japan|        25218|
|       NULL|       Finland|        10666|
|       NULL|        France|       109848|
+-----------+--------------+-------------+
only showing top 5 rows



The rows on top (date is `NULL`) are the country totals across all dates, which the rollup did not have.

`NULL` markers get hard to read with more columns. `grouping_id()` labels each row with the level of aggregation instead: 0 means grouped by both columns, 1 means the country was rolled up, 2 means the date was rolled up, and 3 is the grand total.

In [39]:
from pyspark.sql.functions import grouping_id

dfNoNull.cube("InvoiceDate", "Country")\
  .agg(grouping_id().alias("level"), sum("Quantity"))\
  .groupBy("level").count()\
  .orderBy("level").show()

+-----+-----+
|level|count|
+-----+-----+
|    0| 1682|
|    1|  305|
|    2|   37|
|    3|    1|
+-----+-----+



There are 1,682 date-by-country rows (level 0), 305 daily totals (level 1), 37 country totals (level 2) and one grand total (level 3). A cube is a quick way to build a summary table that others can query at whatever level they need.

## Pivot

A pivot turns the values of one column into columns of their own. Here we want one row per date and one column per country, each holding that country's total quantity.

Spark has to find all distinct countries before it can build the columns, which costs an extra pass over the data. If you already know the values you want, pass them as a list. It is faster, and you only get the columns you asked for:

In [40]:
countries = ["United Kingdom", "France", "Germany", "USA"]

pivoted = df.groupBy("InvoiceDate")\
  .pivot("Country", countries)\
  .agg(sum("Quantity"))\
  .orderBy("InvoiceDate")

pivoted.show(5)

+-----------+--------------+------+-------+----+
|InvoiceDate|United Kingdom|France|Germany| USA|
+-----------+--------------+------+-------+----+
| 2010-12-01|         23949|   449|    117|NULL|
| 2010-12-02|         20873|  NULL|    146|NULL|
| 2010-12-03|         10439|   239|    170|NULL|
| 2010-12-05|         13604|   611|   1362|NULL|
| 2010-12-06|         20669|  NULL|     85|NULL|
+-----------+--------------+------+-------+----+
only showing top 5 rows



Days where a country had no sales show up as `NULL`. With more than one aggregation in `agg()`, Spark creates one column per country and aggregation, named like `France_sum(Quantity)`.

Spark 3.4 added `unpivot` (also called `melt`) to go the other way, from wide back to long:

In [41]:
pivoted.unpivot("InvoiceDate", countries, "Country", "Quantity")\
  .where("Quantity IS NOT NULL")\
  .show(8)

+-----------+--------------+--------+
|InvoiceDate|       Country|Quantity|
+-----------+--------------+--------+
| 2010-12-01|United Kingdom|   23949|
| 2010-12-01|        France|     449|
| 2010-12-01|       Germany|     117|
| 2010-12-02|United Kingdom|   20873|
| 2010-12-02|       Germany|     146|
| 2010-12-03|United Kingdom|   10439|
| 2010-12-03|        France|     239|
| 2010-12-03|       Germany|     170|
+-----------+--------------+--------+
only showing top 8 rows



Finally, free up the memory we used for the cached DataFrame:

In [42]:
df.unpersist()

DataFrame[InvoiceNo: string, StockCode: string, Description: string, Quantity: bigint, InvoiceDate: date, UnitPrice: double, CustomerID: string, Country: string, Revenue: double]